# Example: Let's Explore Binomial Lattice Model Trade Exit Rules
In this example, we will test the binomial lattice model trade exit rules we developed in lecture.
We use a binomial lattice to compute the probability that a terminal benchmark-adjusted return exceeds a declared target over a holding period $T=N\Delta t$, where $N$ is the number of lattice steps. We also compute the complementary probability of finishing at or below that target.

> __Learning Objectives:__
> 
> In this example students will learn to:
> * __Parameter Estimation:__ Estimate binomial lattice parameters from historical stock data. We will compute the up factor, down factor, and probability values using real market data from the S&P 500.
> * __Model Construction:__ Build a binomial equity price tree model using historical parameters. We will construct and populate a lattice model to simulate future stock price movements over a specified time horizon.
> * __Probability Analysis:__ Calculate cumulative probabilities for achieving target returns. We will determine the likelihood of reaching specific return thresholds using binomial probability distributions and analyze multiple return scenarios.

This sounds cool! Let's get started!

___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

Let's set up our code environment:

In [1]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

  Activating 

project at `~/Desktop/julia_work/CHEME-5660-CourseRepository-Fall-2026`


For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/) and the [CHEME 5660 Quantitative Finance Package documentation](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/).

### Data
We gathered daily open-high-low-close (OHLC) data for each firm in the [S&P500](https://en.wikipedia.org/wiki/S%26P_500) from `01-03-2014` until `12-31-2024`, along with data for a few exchange-traded funds and volatility products during that time period. 

Let's load `original_dataset`, a ticker-to-`DataFrame` dictionary, by calling [the `MyTrainingMarketDataSet()` function](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/data/#VLQuantitativeFinancePackage.MyTrainingMarketDataSet) and remove firms that do not have the maximum number of trading days. The cleaned dataset $\mathcal{D}$ will be stored in the `dataset` variable.

In [2]:
original_dataset = MyTrainingMarketDataSet() |> x-> x["dataset"];

Not all tickers in our dataset have the maximum number of trading days for various reasons, e.g., acquisition or delisting events. Let's collect only those tickers with the maximum number of trading days.

First, let's compute the number of records for a firm that we know has the maximum value, e.g., `AAPL`, and save that value in the `maximum_number_trading_days::Int64` variable:

In [3]:
maximum_number_trading_days = original_dataset["AAPL"] |> nrow # nrow? (check out: DataFrames.jl)

2767

Now, let's iterate through our data and collect only tickers with `maximum_number_trading_days` records. We'll save that data in the `dataset::Dict{String,DataFrame}` variable:

In [4]:
dataset = let

    # initialize -
    dataset = Dict{String, DataFrame}();

    # Retain firms that share the complete trading calendar -
    for (ticker, data) ∈ original_dataset
        if (nrow(data) == maximum_number_trading_days)
            dataset[ticker] = data;
        end
    end
    dataset; # return
end;

How many firms do we have with the full number of trading days? Let's use [the `length(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.length), which works for dictionaries in addition to arrays, sets, and other collections.

In [5]:
length(dataset) # tells us how many keys are in the dictionary (how many firms in our dataset?)

424

Finally, let's get a list of the firms in our cleaned dataset and sort them alphabetically. We store the sorted firm ticker symbols in the `list_of_tickers::Array{String,1}` variable.

In [6]:
list_of_tickers = keys(dataset) |> collect |> sort # list of firm "ticker" symbols in alphabetical order

424-element Vector{String}:
 "A"
 "AAL"
 "AAP"
 "AAPL"
 "ABBV"
 "ABT"
 "ACN"
 "ADBE"
 "ADI"
 "ADM"
 "ADP"
 "ADSK"
 "AEE"
 ⋮
 "WST"
 "WU"
 "WY"
 "WYNN"
 "XEL"
 "XOM"
 "XRAY"
 "XYL"
 "YUM"
 "ZBRA"
 "ZION"
 "ZTS"

### Constants
Finally, let's set some constants we'll use later in this notebook. The comments describe the constants, their units, and permissible values.

In [7]:
TSIM = 63; # number of trading days to simulate
Δt = (1.0/252); # step size: 1 trading day in units of years
g_y = 0.05; # continuous annual rate for the selected benchmark yield y (1/years)

___

## Task 1: Estimate lattice parameters from historical data
To create a binomial lattice model for future share prices, we must estimate three critical parameters: $p$, $u$, and $d$.

> __Parameter Definitions__
>
>* The $p$ parameter represents the probability of a share price increase or an `up` move between two consecutive periods $j\rightarrow{j+1}$. Since a binary lattice model only allows `up` and `down` moves, the probability of a `down` move is $1-p$.
>* The $u$ parameter represents the magnitude of an `up` move. If $S_{j}$ denotes the share price in period $j$, and $S_{j+1}$ is the share price in the next period, then an `up` move results in $S_{j+1} = u\cdot{S}_{j}$.
>* The $d$ parameter represents the magnitude of a `down` move. If $S_{j}$ denotes the share price in period $j$, and $S_{j+1}$ is the share price in the next period, then a `down` move results in $S_{j+1} = d\cdot{S}_{j}$.

To start, let's select a firm from the dataset to explore.

In [8]:
selected_firm_ticker = "ADBE" # fixed ticker for reproducibility
selected_firm_index = findfirst(==(selected_firm_ticker), list_of_tickers)
selected_firm_data = dataset[selected_firm_ticker]

Row,volume,volume_weighted_average_price,open,close,high,low,timestamp,number_of_transactions
,Float64,Float64,Float64,Float64,Float64,Float64,DateTime,Int64
1,1.58864e6,59.3388,59.19,59.16,59.685,59.11,2014-01-03T05:00:00,13477
2,3.75391e6,58.2928,58.06,58.12,58.77,58.01,2014-01-06T05:00:00,27434
3,2.96361e6,58.5869,58.26,58.97,59.05,58.06,2014-01-07T05:00:00,22544
4,3.45596e6,58.9278,59.12,58.9,59.28,58.46,2014-01-08T05:00:00,25309
5,2.42743e6,59.157,58.99,59.09,59.53,58.72,2014-01-09T05:00:00,18370
6,1.82906e6,59.3491,59.35,59.53,59.63,58.95,2014-01-10T05:00:00,15386
7,2.92875e6,58.8357,59.28,58.6,59.44,58.41,2014-01-13T05:00:00,24879
8,4.18005e6,60.0286,58.68,60.37,60.53,58.63,2014-01-14T05:00:00,25884
9,4.22034e6,61.4091,60.44,61.68,61.81,60.4,2014-01-15T05:00:00,30992


### Estimate the $u$, $d$, and $p$ parameters from the data
For the selected firm, compute the annualized one-step log-growth vector with `log_growth_matrix(dataset, selected_firm_ticker)`. Positive observations determine representative up factors, negative observations determine representative down factors, and the fraction of positive observations estimates the real-world up probability.

In [9]:
log_growth_array = log_growth_matrix(dataset, selected_firm_ticker) # array holding growth rate time series

2766-element Vector{Float64}:
  -4.48177101721035
   1.2681989501752422
   1.4620645924022435
   0.9782541450394576
   0.816991570922429
  -2.189412070255259
   5.058220052146569
   5.729703164907576
   0.8144832602206464
  -1.3941711674103718
  -1.1683502506700052
   2.0363580890577953
  -2.8152388299854025
   ⋮
 -10.318765952539094
   1.193236717285588
  -4.558776434053768
  -4.775255302688766
  -3.634169535759943
   1.4308473355847118
   1.1050283272038928
  -0.01619378627487112
   1.939109727885214
  -2.805837196438786
  -0.3660345335453977
   0.3514260052939249

Pass the selected firm's `log_growth_array::Vector{Float64}` to `RealWorldBinomialProbabilityMeasure`. The callable type delegates to the vector method that estimates the representative up factor $\bar u$, down factor $\bar d$, and real-world up probability $\bar p$.

In [10]:
(ū,d̄,p̄) = let

    # initialize -
    u = nothing; 
    d = nothing; 
    p = nothing; 

    (u,d,p) = (RealWorldBinomialProbabilityMeasure())(log_growth_array; Δt = Δt)
    
    (u,d,p); # return
end;

Let's create a table for the binomial lattice parameters:

In [11]:
let

    # initialize -
    df = DataFrame();

    row_df = (
        ticker = selected_firm_ticker,
        upfactor = ū,
        downfactor = d̄,
        probability = p̄,
    );
    push!(df, row_df);

      # display the table -
    pretty_table(df, 
         table_format = TextTableFormat(borders = text_table_borders__compact));    
    
end

 -------- ---------- ------------ -------------
  ticker   upfactor   downfactor   probability 
  String    Float64      Float64       Float64 
 -------- ---------- ------------ -------------
    ADBE    1.01165     0.987349      0.556761
 -------- ---------- ------------ -------------


Let's look at the growth-rate distribution for the selected firm:

In [12]:
UnicodePlots.histogram(log_growth_array, nbins=25, closed=:left)

                  ┌                                        ┐ 
   [-45.0, -40.0) ┤▏ 1                                       
   [-40.0, -35.0) ┤  0                                       
   [-35.0, -30.0) ┤▏ 1                                       
   [-30.0, -25.0) ┤▏ 1                                       
   [-25.0, -20.0) ┤▎ 5                                       
   [-20.0, -15.0) ┤▍ 12                                      
   [-15.0, -10.0) ┤▊ 31                                      
   [-10.0,  -5.0) ┤█████▏ 196                                
   [ -5.0,   0.0) ┤█████████████████████████▎ 979            
   [  0.0,   5.0) ┤█████████████████████████████████  1 281  
   [  5.0,  10.0) ┤█████▋ 223                                
   [ 10.0,  15.0) ┤▌ 22                                      
   [ 15.0,  20.0) ┤▍ 11                                      
   [ 20.0,  25.0) ┤▏ 3                                       
                  └                                        ┘ 
        

### Build binomial lattice model using historical parameters
Let's construct an instance of [the `MyBinomialEquityPriceTree` type](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/equity/#VLQuantitativeFinancePackage.MyBinomialEquityPriceTree) using the estimated values for the parameters ($\bar{u}$, $\bar{d}$, $\bar{p}$) from above. 

This will enable us to calculate the prices and probabilities in the tree. We store the populated model in the variable `selected_test_model` for future use.

First, specify the `start_index::Int64` as the trading day index in the dataset, which will serve as the tree's starting point or `L = 0`. For reproducibility, let's hard-code this value as `start_index = 1187`, which corresponds to the trading day `2018-09-19` for our dataset.

In [13]:
start_index = 1187; # DO NOT CHANGE hardcoded start-time for reproducibility
stop_index = start_index + TSIM # TSIM defines the number of trading days to simulate
println("Visualize $(selected_firm_ticker) between trading days ($(start_index) -> $(stop_index))")

Visualize ADBE between trading days (1187 -> 1250)


Next, let's build and populate the binomial lattice model using the estimated parameters and the initial share price $S_{0}$, which we set as the volume-weighted average price (VWAP) on the `start_index` trading day.

We save the populated model in the `selected_test_model::MyBinomialEquityPriceTree` variable:

In [14]:
selected_test_model = let

    # initialize -
    model = nothing;
    Sₒ = selected_firm_data[start_index, :volume_weighted_average_price]; # set the initial share price
    
    
    # Build the forecasting lattice from the estimated real-world parameters -
    model = build(MyBinomialEquityPriceTree, (
        u = ū, d = d̄, p = p̄)) |> (model -> populate(model, Sₒ = Sₒ, h = TSIM));
    
    
    model; # return
end;

In [15]:
typeof(selected_test_model) |> T-> fieldnames(T)

(:u, :d, :p, :μ, :T, :connectivity, :levels, :ΔT, :data)

___

## Task 2: Compute the cumulative probability of achieving a target return
In this task, we compute the probability that the terminal fractional return strictly exceeds $\rho_\star$ over a holding period $T=N\Delta t$.

For $\rho_\star>-1$, the logarithmic derivation in the lecture gives a real-valued count threshold. In code, we classify the actual monotone terminal-node returns instead of blindly rounding that threshold; this preserves the strict inequality when a target is exactly attainable. Because every frictionless terminal price is positive, a target $\rho_\star\le-1$ is cleared with probability one.

In [16]:
ρ₊ = -0.05; # target fractional return over the simulation horizon

Compute the smallest up-move count whose actual terminal return is strictly greater than the target. The helper returns `0` when every node succeeds and `N+1` when no node succeeds.

In [17]:
function strict_lattice_threshold(ρ_star::Real, u::Real, d::Real, N::Int, g_y::Real, Δt::Real)
    @assert u > d > 0 && N >= 0 && Δt > 0
    ρ_star <= -1 && return 0 # every positive-price frictionless return exceeds -1

    # Check the strict predicate at each monotone terminal node; this avoids log-threshold roundoff.
    for k ∈ 0:N
        terminal_return = u^k * d^(N-k) * exp(-g_y*N*Δt) - 1
        terminal_return > ρ_star && return k
    end
    return N + 1 # target lies above every terminal node
end

kmin = strict_lattice_threshold(ρ₊, ū, d̄, TSIM, g_y, Δt)

# Check the domain and an exactly attainable strict boundary.
@assert strict_lattice_threshold(-1.0, ū, d̄, TSIM, g_y, Δt) == 0
u_test, d_test, N_test = 1.015, 0.99, 20
ρ_boundary = u_test^3 * d_test^17 - 1
@assert strict_lattice_threshold(ρ_boundary, u_test, d_test, N_test, 0.0, 1/252) == 4

In [18]:
println("Minimum `up` moves needed to exceed the target $(ρ₊*100)% over $(TSIM) trading days: kmin = $(kmin)")

Minimum `up` moves needed to exceed the target -5.0% over 63 trading days: kmin = 32


We now evaluate the binomial upper tail, handling certain and impossible thresholds explicitly. We store the result in the `test_probability::Float64` variable.

In [19]:
test_probability = let 
    
    # Calculate probability using binomial distribution
    total_periods = TSIM
    distribution = Binomial(total_periods, p̄)
    if kmin <= 0
        1.0
    elseif kmin > total_periods
        0.0
    else
        ccdf(distribution, kmin - 1)
    end
end;

In [20]:
println("Probability of exceeding the target $(ρ₊*100)% over $(TSIM) trading days: P = $(round(test_probability*100, digits=4))%")

Probability of exceeding the target -5.0% over 63 trading days: P = 81.7947%


Let's compute the probability that the fractional return is greater than various target values over the holding period of `TSIM` trading days. We'll store the results in the `P::Array{Float64,2}` array where the first column is the fractional return and the second column is the corresponding probability.

In [21]:
P = let

    # Initialize -
    total_periods = TSIM
    ρ_targets = collect(range(-0.05, stop=0.15, length=9))
    R = Array{Float64,2}(undef, length(ρ_targets), 2) # target and exceedance probability
    distribution = Binomial(total_periods, p̄)

    # Compute the strict terminal probability for each target -
    for idx ∈ eachindex(ρ_targets)
        k_min = strict_lattice_threshold(ρ_targets[idx], ū, d̄, total_periods, g_y, Δt)
        probability = k_min <= 0 ? 1.0 :
            (k_min > total_periods ? 0.0 : ccdf(distribution, k_min - 1))
        R[idx, 1] = ρ_targets[idx]
        R[idx, 2] = probability
    end

    R
end

9×2 Matrix{Float64}:
 -0.05   0.817947
 -0.025  0.743948
  0.0    0.656631
  0.025  0.559853
  0.05   0.459129
  0.075  0.360723
  0.1    0.270522
  0.125  0.192998
  0.15   0.130576

Finally, let's create a table of the results using [the `pretty_table(...)` function from the `PrettyTables.jl` package](https://github.com/ronisbr/PrettyTables.jl).

In [22]:
let

    # initialize -
    df = DataFrame();

    for r ∈ eachrow(P)
        row_df = (
            target_return = r[1],
            P = r[2],
            P̄ = 1.0 - r[2],
        );
        push!(df, row_df);
    end

      # display the table -
    pretty_table(df, 
         table_format = TextTableFormat(borders = text_table_borders__compact));    
end

 --------------- ---------- ----------
  target_return          P          P̄ 
        Float64    Float64    Float64 
 --------------- ---------- ----------
          -0.05   0.817947   0.182053
         -0.025   0.743948   0.256052
            0.0   0.656631   0.343369
          0.025   0.559853   0.440147
           0.05   0.459129   0.540871
          0.075   0.360723   0.639277
            0.1   0.270522   0.729478
          0.125   0.192998   0.807002
           0.15   0.130576   0.869424
 --------------- ---------- ----------


___

## Summary
In this notebook, we estimated a binomial lattice from historical observations and used it to evaluate a strict terminal return target.

> __Key Takeaways:__
>
> * **Historical estimates:** Positive and negative observations determine representative up and down factors; the up count divided by all observations estimates the up probability.
> * **Terminal distribution:** The fixed parameters define a recombining tree with a binomial probability at each terminal node.
> * **Strict target probability:** The return inequality gives an integer up-move threshold. Its binomial upper tail is the model probability of exceeding the target.

The result is a conditional model probability. Its meaning depends on the historical sample, estimator, benchmark, horizon, and fixed-lattice assumptions.

___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products, or any investment or trading advice or strategy, is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.